# Routing

In [11]:
%matplotlib inline
from IPython.display import display
import pandas as pd

from dualde import load_example
from dualde.routing import run_information_transfer_pipeline

EXAMPLE, cfg = load_example()
OUT = EXAMPLE / "outputs"
scores = pd.read_csv(EXAMPLE / "scores" / "phenotype.csv")
coup_path = EXAMPLE / "couplings" / cfg["couplings_file"]
result = run_information_transfer_pipeline(
    all_merged_df=scores,
    couplings_file=str(coup_path),
    functional_residues_file=str(OUT / "functional_residues.csv"),
    active_site_ligands=list(cfg["ligand_names"]),
    activity_wt=cfg["activity_wt"],
    abundance_wt=cfg["abundance_wt"],
    score_col=cfg["routing_score_col"],
    low_score_percentile=cfg["low_score_percentile"],
    prob_threshold=cfg["prob_threshold"],
    n_paths=cfg["n_paths"],
    weighted=True,
    dist_cutoff=cfg["dist_cutoff"],
    functional_dist_cutoff=cfg["functional_dist_cutoff"],
    seq_len=cfg["seq_len"],
    verbose=False,
)
paths, pairs = result["paths_df"], result["pairs_df"]
paths.to_csv(OUT / "coupling_paths.csv", index=False)
pairs.to_csv(OUT / "coupling_pairs.csv", index=False)
long_paths = paths.loc[paths["n_edges"] > cfg["min_n_edges"]].copy()
print(len(paths), "paths", int((pairs["n_paths_found"] > 0).sum()), "pairs")
print(len(long_paths), "paths with more than", cfg["min_n_edges"], "edges")
display(long_paths.nsmallest(8, "n_edges")[["source", "target", "path_rank", "n_edges", "path_str"]])


932 paths 188 pairs
766 paths with more than 5 edges


,source,target,path_rank,n_edges,path_str
6,15,51,2,6,15 -> 17 -> 96 -> 94 -> 92 -> 49 -> 51
15,15,82,1,6,15 -> 17 -> 18 -> 60 -> 97 -> 56 -> 82
28,15,93,4,6,15 -> 17 -> 18 -> 60 -> 97 -> 95 -> 93
46,16,15,2,6,16 -> 56 -> 97 -> 60 -> 18 -> 17 -> 15
55,16,51,1,6,16 -> 56 -> 82 -> 91 -> 92 -> 49 -> 51
68,16,82,4,6,16 -> 56 -> 95 -> 80 -> 79 -> 81 -> 82
77,16,93,3,6,16 -> 56 -> 97 -> 77 -> 80 -> 95 -> 93
205,44,59,1,6,44 -> 47 -> 17 -> 15 -> 14 -> 50 -> 59


In [12]:
test_path = [15, 17, 18, 60, 97, 56, 82]
path_str = " -> ".join(map(str, test_path))
long_paths = long_paths.loc[long_paths["path_str"] == path_str].copy()
if long_paths.empty:
    raise ValueError(f"path not in long_paths: {path_str}")
long_paths = long_paths.sort_values("path_rank").head(1)
print(path_str)

15 -> 17 -> 18 -> 60 -> 97 -> 56 -> 82


In [13]:
# sort by path_cost
long_paths.sort_values("path_cost", ascending=True, inplace=True)
long_paths.head()

,source,target,path_rank,path,path_str,n_edges,path_cost,path_coupling_score,min_edge_prob,mean_edge_prob,edge_probs
15,15,82,1,"[15, 17, 18, 60, 97, 56, 82]",15 -> 17 -> 18 -> 60 -> 97 -> 56 -> 82,6,0.082038,-0.082038,0.959076,0.986327,"[0.9932280867167422, 0.9752958122175348, 0.999..."


In [14]:
import pandas as pd

from dualde.path_tests import (
    result_record,
    test_path_vs_contact_decoys,
    test_path_vs_rsa_matched_contact_decoys,
    test_path_vs_rsa_matched_sets,
)
from dualde.structure_graph import build_contact_graph, residue_distance_map, residue_rsa_map

pdb_path = EXAMPLE / "pdb" / f"{cfg['pdb_id']}_cleaned.pdb"
dist = residue_distance_map(pdb_path, chain_id=cfg["chain"], atom="CA")
scored = set(scores["target_pos"].dropna().astype(int))
rsa_map = {
    int(pos): round(float(value), 3)
    for pos, value in residue_rsa_map(pdb_path, chain_id=cfg["chain"], cap_at_monomer=False).items()
    if int(pos) in scored
}
contact = build_contact_graph(dist, cutoff=cfg["contact_distance_cutoff"])
print(contact.number_of_nodes(), "residues", contact.number_of_edges(), "contacts", len(rsa_map), "RSA")

common = dict(
    df=scores,
    n_null=1000,
    seed=0,
    rsa_map=rsa_map,
    exclude_endpoints=True,
    statistic="both_low",
    alternative="greater",
    activity_wt=cfg["activity_wt"],
    abundance_wt=cfg["abundance_wt"],
    verbose=False,
)
rows = []
record = next(long_paths.itertuples(index=False))
path = test_path
print(record.path_str)
tests = {
    "contact": test_path_vs_contact_decoys(
        path, dist_df=dist, cutoff=cfg["contact_distance_cutoff"], match_buriedness=False, **common
    ),
    "contact_rsa": test_path_vs_rsa_matched_contact_decoys(
        path, dist_df=dist, cutoff=cfg["contact_distance_cutoff"], rsa_tol=0.1, **common
    ),
    "rsa_sets": test_path_vs_rsa_matched_sets(path, rsa_tol=0.1, **common),
}
for res in tests.values():
    row = result_record(res)
    null = res["null_stats"]
    row["z"] = float((res["observed"] - null.mean()) / null.std())
    row["source"] = int(record.source)
    row["target"] = int(record.target)
    row["path_rank"] = int(record.path_rank)
    row["n_edges"] = int(record.n_edges)
    row["path"] = " -> ".join(map(str, path))
    rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "path_null_tests.csv", index=False)
display(summary[["test", "observed", "null_mean", "null_sd", "p_value", "z"]])


156 residues 721 contacts 122 RSA
15 -> 17 -> 18 -> 60 -> 97 -> 56 -> 82


,test,observed,null_mean,null_sd,p_value,z
0,contact,0.440523,0.187787,0.161423,0.079748,1.565673
1,contact_rsa,0.440523,0.272842,0.145945,0.124875,1.148928
2,rsa_sets,0.440523,0.259088,0.109169,0.046953,1.661965
